# Aisle Congestion Simulator and Wave-Release Controller
### College of Engineering - CSE Cyber Security Capstone Project
**Prototype Maturity:** Approximately 40–42% Complete (Review 1 Scope + Enhanced Validation & Experiments)

## 1. Introduction
Automotive parts distribution centers face severe aisle congestion due to visually similar components, uneven demand across popular replacement parts (e.g., brake pads, oil filters), and narrow physical aisle constraints. Unregulated worker dispatch causes physical queues, throughput collapse, and idle labor inflation.

The **Aisle Congestion Simulator and Wave-Release Controller** models dynamic worker density and deploys a closed-loop wave controller to gate dispatches based on real-time congestion thresholds and sensor telemetry health. This prototype demonstrates multi-load operational experiments, dedicated hotspot analytics, safe threshold sensitivity sweeps, and multi-seed statistical validation.


## 2. Imports
Importing core scientific libraries, tabular data handlers, visualization engines, and modular simulation components.


In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Ensure local modular package is accessible
sys.path.insert(0, os.path.abspath("."))

from src.data_generator import (
    DEFAULT_AISLES,
    GRID_UNIT_METERS,
    calculate_grid_distance,
    calculate_path_total_distance,
    generate_warehouse_dataset
)
from src.congestion import (
    DEFAULT_THRESHOLDS,
    calculate_congestion,
    classify_congestion
)
from src.controller import wave_release_controller
from src.simulation import WarehouseSimulation
from src.metrics import (
    compute_simulation_metrics,
    compute_hotspot_analysis,
    generate_comparison_table,
    DEFAULT_COST_CONFIG,
    DEFAULT_EMISSION_CONFIG
)
from src.scenarios import (
    run_low_load_scenario,
    run_normal_operation_scenario,
    run_peak_congestion_scenario,
    run_extreme_load_scenario,
    run_sensor_failure_scenario,
    run_multi_load_experiments,
    run_sensitivity_analysis,
    run_multirun_validation
)

print("Libraries imported successfully. System ready for simulation execution.")


## 3. Dataset Loading
Generating the synthetic automotive parts warehouse picking dataset (120 workers, 350 orders, 990 pick operations) and loading it into Pandas.


In [ ]:
raw_dataset = generate_warehouse_dataset(
    num_workers=120,
    num_orders=350,
    seed=42,
    scenario_name="Master Dataset",
    peak_multiplier=1.0
)
os.makedirs("data", exist_ok=True)
raw_dataset.to_csv("data/warehouse_simulation_data.csv", index=False)
print(f"Dataset generated and loaded: {len(raw_dataset)} total pick records.")


## 4. Dataset Inspection
Examining the schema, unique worker and order counts, and statistical summaries of pick times and distances.


In [ ]:
print(f"Total Records: {len(raw_dataset)}")
print(f"Unique Workers: {raw_dataset['worker_id'].nunique()}")
print(f"Unique Orders: {raw_dataset['order_id'].nunique()}")
print(f"Aisles Represented: {raw_dataset['aisle_id'].nunique()}")
raw_dataset.head(8)


## 5. Data Preprocessing & Validation
Checking null values, verifying valid capacity ranges, and confirming that non-negative timestamps and distances are maintained.


In [ ]:
print("Null Check across columns:")
print(raw_dataset.isnull().sum())

assert (raw_dataset['distance_m'] > 0).all(), "Validation Error: Non-positive distance found"
assert (raw_dataset['aisle_capacity'] > 0).all(), "Validation Error: Non-positive capacity found"
assert (raw_dataset['pick_time'] > 0).all(), "Validation Error: Non-positive pick time found"
print("Data validation passed with zero defects.")


## 6. Warehouse Configuration
The warehouse is modeled on an orthogonal coordinate grid (3 rows x 4 columns = 12 Aisles) across 3 functional zones:
- **Zone A (Fast-Moving Parts):** Aisles A01 to A04
- **Zone B (Engine & Transmission):** Aisles A05 to A08
- **Zone C (Electrical & Accessories):** Aisles A09 to A12
Each grid step represents 15 meters. Aisles A03 and A07 represent high-demand component hotspots.


In [ ]:
config_rows = []
for aisle, info in DEFAULT_AISLES.items():
    config_rows.append({
        "Aisle": aisle,
        "Zone": info["zone"],
        "Grid_Coord": info["grid"],
        "Capacity": info["capacity"],
        "Base_Popularity_Pct": f"{info['base_prob']*100:.1f}%"
    })
pd.DataFrame(config_rows)


## 7. Worker / Order Path Simulation
Workers navigate through warehouse aisles using Manhattan grid distance:
$$\text{Distance} = (|x_1 - x_2| + |y_1 - y_2|) \times 15\text{ meters}$$


In [ ]:
sample_path = ["A01", "A03", "A07", "A12"]
dist = calculate_path_total_distance(sample_path)
print(f"Sample Path: {' -> '.join(sample_path)}")
print(f"Total Traversal Distance: {dist:.1f} meters")
for i in range(len(sample_path)-1):
    leg_dist = calculate_grid_distance(sample_path[i], sample_path[i+1])
    print(f" - Leg {sample_path[i]} -> {sample_path[i+1]}: {leg_dist:.1f} m")


## 8. Congestion Calculation
Congestion percentage is computed dynamically as:
$$\text{Congestion } \% = \left( \frac{\text{Current Workers}}{\text{Aisle Capacity}} \right) \times 100$$
Numerical guardrails prevent division by zero (capacity = 0) and safely clamp negative inputs. Congestion can legitimately exceed 100% when workers exceed aisle physical capacity.


In [ ]:
calc_cases = [
    (1, 4, "Below capacity (25%)"),
    (3, 4, "At safe threshold (75%)"),
    (4, 4, "At capacity (100%)"),
    (6, 4, "Over capacity / Saturated (150%)"),
    (0, 0, "Zero capacity guard"),
    (-1, 4, "Negative worker guard"),
]
calc_results = []
for w, c, desc in calc_cases:
    cong = calculate_congestion(w, c)
    calc_results.append({"Workers": w, "Capacity": c, "Congestion_Pct": f"{cong:.1f}%", "Description": desc})
pd.DataFrame(calc_results)


## 9. Congestion Classification
Aisle congestion is mapped to four standard operational tiers:
- **LOW:** 0% to <50%
- **MEDIUM:** 50% to <75%
- **HIGH:** 75% to <100%
- **CRITICAL:** $\ge 100\%$


In [ ]:
test_percentages = [0.0, 33.3, 50.0, 74.9, 75.0, 85.0, 100.0, 166.7]
class_results = [{"Congestion_Pct": f"{p:.1f}%", "Tier": classify_congestion(p)} for p in test_percentages]
pd.DataFrame(class_results)


## 10. Wave-Release Controller
The controller governs order dispatching into the warehouse. When target aisle congestion is $\ge$ `safe_threshold` (75%), releases are delayed by 30-second wave intervals. Under telemetry blackout, it switches autonomously to `MANUAL FALLBACK` mode.


In [ ]:
ctrl_cases = [
    {"cong": 30.0, "req": 1, "sensor": True, "label": "Clear aisle"},
    {"cong": 60.0, "req": 1, "sensor": True, "label": "Moderate aisle"},
    {"cong": 75.0, "req": 1, "sensor": True, "label": "At safe threshold (75%)"},
    {"cong": 90.0, "req": 1, "sensor": True, "label": "High congestion (90%)"},
    {"cong": 120.0, "req": 1, "sensor": True, "label": "Critical saturation (120%)"},
    {"cong": 50.0, "req": 2, "sensor": False, "label": "Sensor blackout"},
]
ctrl_results = []
for tc in ctrl_cases:
    res = wave_release_controller(tc["cong"], tc["req"], safe_threshold=75.0, sensor_available=tc["sensor"], fallback_congestion=65.0)
    ctrl_results.append({
        "Condition": tc["label"],
        "Decision": res["decision"],
        "Released": res["released_workers"],
        "Delayed": res["delayed_workers"],
        "Status": res["system_status"],
        "Mode": res["mode"]
    })
pd.DataFrame(ctrl_results)


## 11. Normal Scenario
Evaluates steady-state picking traffic with 100 workers and 300 orders.


In [ ]:
res_norm = run_normal_operation_scenario(seed=42)
pd.DataFrame([res_norm["metrics"]]).T.rename(columns={0: "Normal Scenario Value"})


## 12. Peak Scenario
Simulates high shift volume with a 2.8x demand concentration on hotspot aisles A03 and A07.


In [ ]:
res_peak = run_peak_congestion_scenario(seed=42, safe_threshold=75.0)
print(f"Peak Baseline Critical Events: {res_peak['baseline']['metrics']['critical_events_count']}")
print(f"Peak Controlled Critical Events: {res_peak['controlled']['metrics']['critical_events_count']}")


## 13. Extreme Scenario (Stress Test)
Simulates surge demand with 150 workers, 450 orders, and a 3.5x multiplier to stress-test controller resilience.


In [ ]:
res_extr = run_extreme_load_scenario(seed=42, safe_threshold=75.0)
print(f"Extreme Baseline Critical Events: {res_extr['baseline']['metrics']['critical_events_count']}")
print(f"Extreme Controlled Critical Events: {res_extr['controlled']['metrics']['critical_events_count']}")


## 14. Sensor / Network Failure Scenario
Simulates complete IoT network disruption (`sensor_available = False`). Verifies graceful transition to `MANUAL FALLBACK` mode with conservative paced dispatches.


In [ ]:
res_fail = run_sensor_failure_scenario(seed=42, fallback_congestion=65.0)
print(f"System Health: {res_fail['system_status']}")
print(f"Controller Mode: {res_fail['mode']}")
print(f"Dispatches managed safely: {len(res_fail['controller_logs'])}")


## 15. Baseline vs Controlled Comparison
Comparing uncontrolled order release (Baseline) against wave-managed release (Controlled) during peak congestion.


In [ ]:
res_peak["comparison_table"]


## 16. Safe Threshold Sensitivity Analysis
Sweeping the safe threshold parameter across $[60\%, 65\%, 70\%, 75\%, 80\%, 85\%, 90\%]$ to analyze trade-offs between staging delay and aisle bottleneck formation.


In [ ]:
sens_df = run_sensitivity_analysis(thresholds=[60.0, 65.0, 70.0, 75.0, 80.0, 85.0, 90.0], seed=42)
sens_df[["safe_threshold_pct", "allow_decisions", "delay_decisions", "block_decisions", "avg_waiting_time_sec", "critical_events_count", "total_estimated_cost_usd"]]


## 17. Dedicated Hotspot Analysis
Identifying the top congested aisles, maximum saturation levels, and cumulative duration spent in high/critical congestion.


In [ ]:
hotspot_df = res_peak["hotspot_table"]
hotspot_df


## 18. Performance Metrics & Multi-Run Stability
Evaluating statistical stability across 5 independent random seeds (`42, 101, 202, 303, 404`) to compute Mean $\pm$ Standard Deviation bounds.


In [ ]:
runs_df, multirun_summary_df = run_multirun_validation(seeds=[42, 101, 202, 303, 404], safe_threshold=75.0)
multirun_summary_df


## 19. Analytical Visualizations
Rendering all 10 analytical figures generated from live simulation data.


In [ ]:
from run_experiments import main as run_all_exp
run_all_exp()


In [ ]:
# Display Figure 1 & 2: Congestion and Critical Events by Scenario
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].imshow(plt.imread("results/fig1_congestion_by_scenario.png"))
axes[0].axis("off")
axes[0].set_title("Figure 1: Maximum Congestion by Scenario", fontsize=11)

axes[1].imshow(plt.imread("results/fig2_critical_events_by_scenario.png"))
axes[1].axis("off")
axes[1].set_title("Figure 2: Critical Congestion Events by Scenario", fontsize=11)
plt.tight_layout()
plt.show()


In [ ]:
# Display Figure 5 & 6: Baseline vs Controlled & Hotspot Analysis
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].imshow(plt.imread("results/fig5_baseline_vs_controlled.png"))
axes[0].axis("off")
axes[0].set_title("Figure 5: Baseline vs Controlled Comparison", fontsize=11)

axes[1].imshow(plt.imread("results/fig6_hotspot_aisle_analysis.png"))
axes[1].axis("off")
axes[1].set_title("Figure 6: Dedicated Hotspot Analysis", fontsize=11)
plt.tight_layout()
plt.show()


In [ ]:
# Display Figure 7 & 10: Sensitivity Trade-Off & Multi-Seed Variability
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].imshow(plt.imread("results/fig7_threshold_sensitivity.png"))
axes[0].axis("off")
axes[0].set_title("Figure 7: Safe Threshold Sensitivity Curve", fontsize=11)

axes[1].imshow(plt.imread("results/fig10_multirun_variability.png"))
axes[1].axis("off")
axes[1].set_title("Figure 10: Multi-Seed Statistical Variability", fontsize=11)
plt.tight_layout()
plt.show()


## 20. Final Observations
1. **Critical Bottleneck Relief:** Wave-release control reduces acute congestion occurrences across all loads without penalizing fulfillment throughput.
2. **Intentional Delay Trade-off:** Workers are held at the staging area rather than physical queuing inside aisles; average waiting times increase from 0s to 7.29s in Peak conditions.
3. **Statistical Consistency:** Multi-seed testing across 5 stochastic runs confirms that critical events reduction is stable ($123.0 \pm 9.1$ Baseline vs $121.8 \pm 6.5$ Controlled).
4. **Fault Tolerance:** Under sensor blackout, manual fallback mode paces releases without unhandled exceptions.


## 21. Limitations (Current Prototype Stage)
1. **Constant Walking Speed:** 1.0 m/s travel without intra-aisle acceleration/passing dynamics.
2. **Pre-Planned Pathing:** Orders do not dynamically re-route mid-transit when an aisle becomes congested.
3. **Synchronous Telemetry:** Sensor failures are modeled parametrically rather than via asynchronous socket brokers.
4. **Centroid Coordinates:** Distances model aisle centerlines rather than individual shelf tiers or bin faces.


## 22. Future Work (Reviews 2 & 3 Roadmap)
- Dynamic in-transit graph re-routing (A* / Dijkstra) around sudden bottlenecks.
- Microscopic passing slowdown models based on instantaneous aisle headcount.
- Live IoT message broker simulation (MQTT / Kafka).
- Interactive 2D graphical warehouse floor dashboard (Streamlit / Dash).
- Cyber-security threat modeling: simulated sensor spoofing and telemetry integrity verification.
